In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import mlflow
import matplotlib.pyplot as plt
%matplotlib widget

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from sysid.data import DataNormalizer

print(f"Project root: {project_root}")


Project root: /Users/jack/Documents/01_Git/01_promotion/genSecSysId/python


In [2]:
# Set MLflow tracking URI
mlflow_uri = project_root / "mlruns"
mlflow.set_tracking_uri(f"file://{mlflow_uri}")
# mlflow_uri = 'http://mlflowui.informatik.uni-stuttgart.de/'
# mlflow.set_tracking_uri(mlflow_uri)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

# experiment_name = 'duffing-soft-8'
experiment_name = 'crnn-duffing_dev'
# Ranking metric for the summary tables / best-HP-group selection.
eval_metric = 'id/conv/eval_nrmse'

MLflow tracking URI: file:///Users/jack/Documents/01_Git/01_promotion/genSecSysId/python/mlruns


In [3]:
from sysid.models.factory import load_model
from sysid.config import Config
from sysid.data import DataNormalizer

run_id = '89b4a178b38446e5a653c0b28c8c46fd'
print(f"Loading model from run {run_id}...")

# Models are logged as raw .pt files under artifact path "models" (not "model").
# mlflow.pytorch.log_model was never called, so mlflow.pytorch.load_model won't work.
outputs_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="outputs")
models_dir  = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="models")

config = Config.from_yaml(os.path.join(outputs_dir, "config.yaml"))
model  = load_model(os.path.join(models_dir, "best_model.pt"), config)

normalizer_path = mlflow.artifacts.download_artifacts(
    run_id=run_id, artifact_path="models/normalizer.json"
)
normalizer = DataNormalizer.load(normalizer_path)



print(f"Model loaded: {type(model).__name__}")
print(model)

print(f"\nNormalizer loaded: {type(normalizer).__name__}")

Loading model from run 89b4a178b38446e5a653c0b28c8c46fd...


Model loaded: SimpleLureSafe
SimpleLureSafe(
  (lure): LureSystemSafe(
    (Delta): DznActivation()
  )
)

Normalizer loaded: DataNormalizer


In [12]:
print(f"Constraints satisfied: {model.check_constraints()}")

A = model.A.detach().numpy()
B = model.B.detach().numpy()
B2 = model.B2.detach().numpy()

C = model.C.detach().numpy()
D = model.D.detach().numpy()
D12 = model.D12.detach().numpy()

C2 = model.C2.detach().numpy()
D21 = model.D21.detach().numpy()

P = model.P.detach().numpy()
L = model.L.detach().numpy()
la = model.la.detach().numpy()

M = np.diag(la)

tau = model.tau.detach().numpy()
alpha = 1.0 / (1.0 + np.exp(-tau))  # sigmoid
s = model.s.detach().numpy()
X = np.linalg.inv(P)  # P^(-1)

nd, nx, nz = model.nd, model.nx, model.nz
ne, nd = model.ne, model.nd

Constraints satisfied: True


In [8]:
def np_bmat(mat):
    mat_list = []
    for col in mat:
        mat_list.append(np.hstack(col))
    return np.vstack(mat_list)

In [9]:
F = np_bmat(
    [
        [-(alpha**2) * P, np.zeros((nx, nd)), P @ C2.T + L.T, P @ A.T],
        [np.zeros((nd, nx)), -np.eye(nd), D21.T, B.T],
        [C2 @ P + L, D21, -2 * M, M @ B2.T],
        [A @ P, B, B2 @ M, -P],
    ]
)

Gs = []
for i in range(nz):
    li = L[i, :].reshape((1, -1), order="C")
    locality_lmi = np_bmat([[(1/s**2).reshape((1,1)), li], [li.T, P]])
    Gs.append(locality_lmi)

In [10]:
# check constraints verification
# F < 0 -> max eigenvalue should be < 0
max_eig_F = np.max(np.linalg.eigvals(F))
print(f"Maximum eigenvalue of F: {max_eig_F:.6e}")
# G_i > 0 -> min eigenvalue should be > 0
for i, G in enumerate(Gs):
    min_eig_G = np.min(np.linalg.eigvals(G))
    print(f"Minimum eigenvalue of G[{i}]: {min_eig_G:.6e}")

Maximum eigenvalue of F: -4.845188e-02
Minimum eigenvalue of G[0]: 3.638680e-02
Minimum eigenvalue of G[1]: 5.119844e-02
Minimum eigenvalue of G[2]: 5.431676e-02
Minimum eigenvalue of G[3]: 3.374592e-02
Minimum eigenvalue of G[4]: 5.162895e-02
Minimum eigenvalue of G[5]: 4.362689e-02
Minimum eigenvalue of G[6]: 2.841570e-02
Minimum eigenvalue of G[7]: 3.877623e-02
Minimum eigenvalue of G[8]: 4.206422e-02
Minimum eigenvalue of G[9]: 5.343512e-02
Minimum eigenvalue of G[10]: 3.623873e-02
Minimum eigenvalue of G[11]: 5.279688e-02
Minimum eigenvalue of G[12]: 5.200432e-02
Minimum eigenvalue of G[13]: 5.081472e-02
Minimum eigenvalue of G[14]: 2.380471e-02
Minimum eigenvalue of G[15]: 2.919916e-02
Minimum eigenvalue of G[16]: 4.273903e-02
Minimum eigenvalue of G[17]: 5.177155e-02
Minimum eigenvalue of G[18]: 4.756278e-02
Minimum eigenvalue of G[19]: 4.484278e-02


In [11]:
# load training data
from sysid.data.direct_loader import load_csv_folder

data_type = 'train'
training_data_directory = os.path.expanduser(os.path.join('~/genSecSysId-Data/data/Duffing/id', data_type))
train_inputs, train_outputs, _, _ = load_csv_folder(
        folder_path=str(training_data_directory),
        input_col=getattr(config.data, "input_col", ["d"]),
        output_col=getattr(config.data, "output_col", ["e"]),
        state_col = None,
        pattern=getattr(config.data, "pattern", "*.csv"),
    )
train_inputs, train_outputs = (
    np.stack(train_inputs),
    np.stack(train_outputs)
)
print(f'train inputs shape: {train_inputs.shape}, train_outputs shape: {train_outputs.shape}')


train inputs shape: (60, 4000, 1), train_outputs shape: (60, 4000, 1)


In [14]:
y_max_train = float(np.nanmax(np.abs(train_outputs)))

In [17]:
import cvxpy as cp
output_std = float(normalizer.output_std.squeeze())
EPS = 1.0e-8

y_max = y_max_train
s_list = np.linspace(1, 40, 10)

for s in s_list:
    constraints = []
    P = cp.Variable((nx, nx), symmetric=True)
    la = cp.Variable((nz, 1))
    M = cp.diag(la)
    L = cp.Variable((nz, nx))

    for li in L:
        li = li.reshape((1, -1), "C")
        constraints.append(
            cp.bmat(
                [
                    [np.array([[1 / s**2]]), li],
                    [li.T, P],
                ]
            )
            >> EPS * np.eye(nx + 1)
        )

    F = cp.bmat(
        [
            [-(alpha**2) * P, np.zeros((nx, nd)), P @ C2.T + L.T, P @ A.T],
            [np.zeros((nd, nx)), -np.eye(nd), D21.T, B.T],
            [C2 @ P + L, D21, -2 * M, M @ B2.T],
            [A @ P, B, B2 @ M, -P],
        ]
    )
    nF = F.shape[0]
    constraints.append(F << -EPS * np.eye(nF))

    constraints.append((s*output_std)**2 * C @ P @ C.T - y_max**2 * np.eye(nd) >> EPS * np.eye(nd))

    objective = cp.Minimize(cp.trace((s*output_std)**2 * C @ P @ C.T))
    problem = cp.Problem(objective, constraints)
    try:
        problem.solve(solver=cp.MOSEK, verbose=False)
    except:
        print(f"Error occurred while solving problem for s={s:.4f}. Skipping this value.")
        continue
    if problem.status != cp.OPTIMAL:
        print(f"Problem not solved to optimality for s={s:.4f}. Status: {problem.status}")
        continue

    print(f'Problem status for s={s:.4f}: {problem.status}')

    
    

Problem status for s=1.0000: optimal
Problem status for s=5.3333: optimal
Problem status for s=9.6667: optimal
Problem status for s=14.0000: optimal
Problem status for s=18.3333: optimal
Problem status for s=22.6667: optimal
Problem status for s=27.0000: optimal
Problem status for s=31.3333: optimal
Problem status for s=35.6667: optimal
Problem status for s=40.0000: optimal
